# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [2]:
%uv pip install scanpy==1.11.5 anndata==0.11.4 pandas==2.3.3 numpy==2.2.6 scikit-learn==1.7.2 umap-learn==0.5.12 leidenalg==0.11.0 igraph==1.0.0

Using Python 3.12.6 environment at: /usr/local
Resolved 38 packages in 212ms
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
texttable  ------------------------------ 10.52 KiB/10.52 KiB
⠙ Preparing packages... (0/14)
texttable  ------------------------------ 10.52 KiB/10.52 KiB
⠙ Preparing packages... (0/14)
texttable  ------------------------------ 10.52 KiB/10.52 KiB
pynndescent ------------------------------ 14.84 KiB/71.79 KiB
⠙ Preparing packages... (0/14)
texttable  ------------------------------ 10.52 KiB/10.52 KiB
pynndescent ------------------------------ 14.84 KiB/71.79 KiB
⠙ Preparing packages... (0/14)
legacy-api-wrap ------------------------------     0 B/9.94 KiB
texttable  ------------------------------ 10.52 KiB/10.52 KiB
pynndescent ------------------------------ 14.84 KiB/71.79 KiB
⠙ Preparing packages... (0/14)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
texttable  ------------------------------ 10.52

In [3]:
import pandas as pd
import numpy as np
import scanpy as sc
import os,gc, time
from tqdm import tqdm

sc.settings.verbosity = 2

timepoints = ['12h', '1d', '2d', '4d', '1w', '2w']

In [4]:
data_path ="/mnt/scrna-chop-data/"

In [5]:
adata = sc.read_h5ad(data_path + "adata_combined_raw.h5ad")

#QC
print("Running QC...", flush=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

print(f"After QC: {adata.shape[0]} cells x {adata.shape[1]} genes", flush=True)
print(f"Timepoint distribution after QC:\n{adata.obs['timepoint'].value_counts().sort_index()}", flush=True)

Running QC...
filtered out 769 cells that have less than 200 genes expressed
filtered out 12846 genes that are detected in less than 3 cells
After QC: 105477 cells x 27944 genes
Timepoint distribution after QC:
timepoint
ctrl    17887
12h     14720
1d      14627
2d      13906
4d      17216
1w      13668
2w      13453
Name: count, dtype: int64


In [ ]:
print("Normalizing...", flush=True)
adata.layers['counts'] = adata.X.copy()  # preserve raw counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print("Normalization complete", flush=True)

adata.write_h5ad("adata_normalized.h5ad")

In [ ]:
#MEMORY HEAVY STEP: IF RUNNING THIS NOTEBOOK NEEDS AT LEAST 20GB OF RAM
deg_results = {}

for tp in timepoints:
    t0 = time.time()
    print(f"\n=== DE: {tp} vs ctrl ===", flush=True)
    # Subset to just this timepoint + control
    mask = adata.obs['timepoint'].isin([tp, 'ctrl'])
    adata_sub = adata[mask].copy()
    print(f"  Subset: {adata_sub.shape[0]} cells ({tp}: {(adata_sub.obs['timepoint']==tp).sum()}, ctrl: {(adata_sub.obs['timepoint']=='ctrl').sum()})", flush=True)
    
    sc.tl.rank_genes_groups(adata_sub, groupby='timepoint', groups=[tp], reference='ctrl',
                            method='wilcoxon', corr_method='benjamini-hochberg',
                            pts=True, use_raw=False)
    
    # Extract results
    result = sc.get.rank_genes_groups_df(adata_sub, group=tp)
    pts_ref = adata_sub.uns['rank_genes_groups']['pts'][['ctrl']].reset_index()
    pts_ref.columns = ['gene', f'pct_ctrl_{tp}']
    pts_ref['gene'] = pts_ref['gene'].astype(str)
    
    result = result.rename(columns={
        'names': 'gene',
        'logfoldchanges': f'log2FC_{tp}',
        'pvals': f'pval_{tp}',
        'pvals_adj': f'padj_{tp}',
        'pct_nz_group': f'pct_injured_{tp}',
    })
    result['gene'] = result['gene'].astype(str)
    result = result.merge(pts_ref, on='gene', how='left')
    deg_results[tp] = result
    n_up = ((result[f'padj_{tp}'] < 0.05) & (result[f'log2FC_{tp}'] > 1)).sum()
    print(f"  {tp}: {n_up} upregulated (padj<0.05, log2FC>1) ({time.time()-t0:.1f}s)", flush=True)
    del adata_sub
    gc.collect()

In [ ]:
#Merged tables
print("\nMerging DEG tables...", flush=True)
merged = deg_results['12h'][['gene', 'log2FC_12h', 'pval_12h', 'padj_12h', 'pct_injured_12h', 'pct_ctrl_12h']]
for tp in ['1d', '2d', '4d', '1w', '2w']:
    r = deg_results[tp][['gene', f'log2FC_{tp}', f'pval_{tp}', f'padj_{tp}', f'pct_injured_{tp}', f'pct_ctrl_{tp}']]
    merged = merged.merge(r, on='gene', how='outer')

In [ ]:
# Compute pseudobulk mean expression per timepoint
print("Computing pseudobulk mean expression...", flush=True)
mean_expr = {}
for tp in ['ctrl'] + timepoints:
    mask = adata.obs['timepoint'] == tp
    if mask.sum() > 0:
        mean_expr[tp] = np.asarray(adata[mask].X.mean(axis=0)).flatten()

mean_df = pd.DataFrame(mean_expr, index=adata.var_names).reset_index().rename(columns={'index': 'gene'})
mean_df.columns = ['gene'] + [f'mean_log_{tp}' for tp in ['ctrl'] + timepoints]
mean_df['gene'] = mean_df['gene'].astype(str)

merged = merged.merge(mean_df, on='gene', how='left')

In [ ]:
#Save merged table
merged.to_csv("results/deg_all_timepoints.csv", index=False)
print(f"\nSaved DEG table: {merged.shape[0]} genes x {merged.shape[1]} columns", flush=True)

In [7]:
merged = pd.read_csv("/mnt/scrna-chop-data/results/deg_all_timepoints.csv")

In [8]:
#summary tables
print("\n=== Summary: upregulated genes (padj<0.05, log2FC>1) ===", flush=True)
for tp in timepoints:
    n_up = ((merged[f'padj_{tp}'] < 0.05) & (merged[f'log2FC_{tp}'] > 1)).sum()
    print(f"  {tp}: {n_up} upregulated genes")
    
# Top 10 at 2d (peak injury response)
print("\n=== Top 10 upregulated at 2d (peak) ===", flush=True)
tp = '2d'
top = merged[(merged[f'padj_{tp}'] < 0.05) & (merged[f'log2FC_{tp}'] > 1)].sort_values(f'log2FC_{tp}', ascending=False).head(10)
for _, row in top.iterrows():
    print(f"  {row['gene']:15s} log2FC={row[f'log2FC_{tp}']:.2f} padj={row[f'padj_{tp}']:.2e} pct_inj={row[f'pct_injured_{tp}']:.3f} pct_ctrl={row[f'pct_ctrl_{tp}']:.3f}")


=== Summary: upregulated genes (padj<0.05, log2FC>1) ===
  12h: 218 upregulated genes
  1d: 626 upregulated genes
  2d: 1802 upregulated genes
  4d: 1055 upregulated genes
  1w: 1143 upregulated genes
  2w: 1340 upregulated genes

=== Top 10 upregulated at 2d (peak) ===
  Krt42           log2FC=24.62 padj=5.23e-14 pct_inj=0.050 pct_ctrl=0.000
  Gm29374         log2FC=23.87 padj=2.76e-06 pct_inj=0.032 pct_ctrl=0.000
  Colec10         log2FC=23.59 padj=6.70e-04 pct_inj=0.024 pct_ctrl=0.000
  Cdc42ep1        log2FC=23.25 padj=3.17e-03 pct_inj=0.021 pct_ctrl=0.000
  Hoxb9           log2FC=22.85 padj=4.06e-02 pct_inj=0.016 pct_ctrl=0.000
  Sprr1a          log2FC=11.94 padj=0.00e+00 pct_inj=0.251 pct_ctrl=0.000
  Dnmt3l          log2FC=11.26 padj=3.41e-12 pct_inj=0.047 pct_ctrl=0.000
  Ecel1           log2FC=9.07 padj=2.80e-273 pct_inj=0.232 pct_ctrl=0.001
  AC154039.2      log2FC=9.05 padj=1.84e-15 pct_inj=0.053 pct_ctrl=0.000
  Gm14206         log2FC=7.63 padj=2.27e-09 pct_inj=0.040 pct_c

In [9]:
#temporal analysis
# Fill NaN log2FC with 0 (genes not tested at a timepoint)
for tp in timepoints:
    merged[f'log2FC_{tp}'] = merged[f'log2FC_{tp}'].fillna(0)
    merged[f'padj_{tp}'] = merged[f'padj_{tp}'].fillna(1.0)
    merged[f'pct_injured_{tp}'] = merged[f'pct_injured_{tp}'].fillna(0)
    merged[f'pct_ctrl_{tp}'] = merged[f'pct_ctrl_{tp}'].fillna(0)

# Define significance: padj < 0.05 AND log2FC > 1
INDUCED_THRESHOLD = 1.0   # log2FC > 1
NOT_INDUCED_THRESHOLD = 0.5  # log2FC < 0.5

for tp in timepoints:
    merged[f'induced_{tp}'] = (merged[f'padj_{tp}'] < 0.05) & (merged[f'log2FC_{tp}'] > INDUCED_THRESHOLD)
    merged[f'not_induced_{tp}'] = merged[f'log2FC_{tp}'] < NOT_INDUCED_THRESHOLD

# Classify temporal profiles
# Early = 12h or 1d; Late = 1w or 2w
early_induced = merged['induced_12h'] | merged['induced_1d']
late_induced = merged['induced_1w'] | merged['induced_2w']
early_not_induced = merged['not_induced_12h'] & merged['not_induced_1d']
late_not_induced = merged['not_induced_1w'] & merged['not_induced_2w']

# Early + sustained: induced early AND induced late
merged['profile_early_sustained'] = early_induced & late_induced

# Early + transient: induced early AND NOT induced late (log2FC < 0.5 at both 1w and 2w)
merged['profile_early_transient'] = early_induced & late_not_induced

# Late onset: NOT induced early AND induced late
merged['profile_late_onset'] = early_not_induced & late_induced

# Must be upregulated at at least one timepoint
any_induced = merged[[f'induced_{tp}' for tp in timepoints]].any(axis=1)
merged['profile_mixed'] = any_induced & ~merged['profile_early_sustained'] & ~merged['profile_early_transient'] & ~merged['profile_late_onset']

In [10]:
df = merged

# Summary
print("\n=== Temporal Profile Classification ===", flush=True)
print(f"  Early + sustained: {df['profile_early_sustained'].sum()} genes")
print(f"  Early + transient: {df['profile_early_transient'].sum()} genes")
print(f"  Late onset:        {df['profile_late_onset'].sum()} genes")
print(f"  Mixed (other):     {df['profile_mixed'].sum()} genes")
print(f"  Total upregulated: {any_induced.sum()} genes")


=== Temporal Profile Classification ===
  Early + sustained: 319 genes
  Early + transient: 256 genes
  Late onset:        491 genes
  Mixed (other):     1584 genes
  Total upregulated: 2650 genes


In [ ]:
# Assign single profile label
df['temporal_profile'] = 'none'
df.loc[df['profile_early_sustained'], 'temporal_profile'] = 'early_sustained'
df.loc[df['profile_early_transient'], 'temporal_profile'] = 'early_transient'
df.loc[df['profile_late_onset'], 'temporal_profile'] = 'late_onset'
df.loc[df['profile_mixed'], 'temporal_profile'] = 'mixed'

# Save intermediate
df.to_csv("results/deg_classified.csv", index=False)
print(f"\nSaved classified DEG table: {df.shape}", flush=True)

# Show some examples from each profile
for profile in ['early_sustained', 'early_transient', 'late_onset']:
    print(f"\n=== Top 5 {profile} by max log2FC ===", flush=True)
    sub = df[df[f'profile_{profile}']].copy()
    sub['max_log2FC'] = sub[[f'log2FC_{tp}' for tp in timepoints]].max(axis=1)
    sub = sub.sort_values('max_log2FC', ascending=False).head(5)
    for _, row in sub.iterrows():
        fcs = " ".join([f"{row[f'log2FC_{tp}']:6.1f}" for tp in timepoints])
        print(f"  {row['gene']:15s} [{fcs}]  ctrl_pct={row['pct_ctrl_2d']:.3f}")

In [42]:
#start from classified
df = pd.read_csv('/mnt/scrna-chop-data/results/deg_classified.csv')
df.columns

Index(['gene', 'log2FC_12h', 'pval_12h', 'padj_12h', 'pct_injured_12h',
       'pct_ctrl_12h', 'log2FC_1d', 'pval_1d', 'padj_1d', 'pct_injured_1d',
       'pct_ctrl_1d', 'log2FC_2d', 'pval_2d', 'padj_2d', 'pct_injured_2d',
       'pct_ctrl_2d', 'log2FC_4d', 'pval_4d', 'padj_4d', 'pct_injured_4d',
       'pct_ctrl_4d', 'log2FC_1w', 'pval_1w', 'padj_1w', 'pct_injured_1w',
       'pct_ctrl_1w', 'log2FC_2w', 'pval_2w', 'padj_2w', 'pct_injured_2w',
       'pct_ctrl_2w', 'mean_log_ctrl', 'mean_log_12h', 'mean_log_1d',
       'mean_log_2d', 'mean_log_4d', 'mean_log_1w', 'mean_log_2w',
       'induced_12h', 'not_induced_12h', 'induced_1d', 'not_induced_1d',
       'induced_2d', 'not_induced_2d', 'induced_4d', 'not_induced_4d',
       'induced_1w', 'not_induced_1w', 'induced_2w', 'not_induced_2w',
       'profile_early_sustained', 'profile_early_transient',
       'profile_late_onset', 'profile_mixed', 'temporal_profile'],
      dtype='object')

In [ ]:
# Only consider genes in one of the three clean profiles
profiles = ['early_sustained', 'early_transient', 'late_onset']
df_candidates = df[df['temporal_profile'].isin(profiles)].copy()
print(f"Candidate genes (3 profiles): {len(df_candidates)}", flush=True)

# Save full clean table
df_candidates.to_csv("/mnt/scrna-chop-data/results/deg_temporal_all.csv", index=False)
print(f"\nSaved cleaned table: {df_candidates.shape[0]} genes", flush=True)

df_temporal = df_candidates

Candidate genes (3 profiles): 1066

Saved ranked table: 1066 genes


In [14]:
df_st1 = pd.read_csv('mcp1_injury_response.csv')

In [15]:
glial_up_genes = set()
glial_up_by_type = {}
for col in ['Astrocyte.up', 'Microglia.up', 'Muller.glia.up']:
    genes = set(df_st1[col].dropna().tolist())
    glial_up_genes.update(genes)
    glial_up_by_type[col.replace('.up', '')] = genes

In [16]:
print(f"Total unique glial upregulated genes: {len(glial_up_genes)}")
print(f"  Astrocyte: {len(glial_up_by_type['Astrocyte'])}")
print(f"  Microglia: {len(glial_up_by_type['Microglia'])}")
print(f"  Muller.glia: {len(glial_up_by_type['Muller.glia'])}")

Total unique glial upregulated genes: 149
  Astrocyte: 61
  Microglia: 77
  Muller.glia: 49


In [17]:
# Flag genes that are also upregulated in glia
df_ranked['glial_upregulated'] = df_ranked['gene'].isin(glial_up_genes)
df_ranked['glial_astrocyte_up'] = df_ranked['gene'].isin(glial_up_by_type['Astrocyte'])
df_ranked['glial_microglia_up'] = df_ranked['gene'].isin(glial_up_by_type['Microglia'])
df_ranked['glial_muller_up'] = df_ranked['gene'].isin(glial_up_by_type['Muller.glia'])

# RGC-specific = NOT upregulated in any glial type
df_ranked['RGC_specific_non_glial'] = ~df_ranked['glial_upregulated']

In [22]:
#summary of ranking for rgc specific non glial
print(f"\n=== RGC Specificity ===")
print(f"  RGC-specific: {df_temporal['RGC_specific_non_glial'].sum()} / {len(df_temporal)}")

# Per profile
for profile in ['early_sustained', 'early_transient', 'late_onset']:
    sub = df_temporal[df_temporal['temporal_profile'] == profile]
    n_spec = sub['RGC_specific_non_glial'].sum()
    print(f"  {profile}: {n_spec}/{len(sub)} RGC-specific")


=== RGC Specificity ===
  RGC-specific: 999 / 1066
  early_sustained: 272/319 RGC-specific
  early_transient: 249/256 RGC-specific
  late_onset: 478/491 RGC-specific


In [24]:
# Save updated ranked table
df_temporal.to_csv("/mnt/scrna-chop-data/results/deg_temporal_with_specificity.csv", index=False)
print(f"\nSaved with specificity annotations: {df_temporal.shape}")



Saved with specificity annotations: (1066, 66)
